## 과제 A : SNS 댓글 모더레이션


In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain_core.output_parsers import PydanticOutputParser


class CommentModeration(BaseModel):
    toxicity: Literal["safe", "warning", "toxic"] = Field(
        description="댓글의 악성 정도. safe는 안전, warning은 주의가 필요한 댓글, toxic은 악성 댓글"
    )

    contains_profanity: bool = Field(
        description="욕설이나 비속어가 포함되어 있는지 여부"
    )

    contains_personal_attack: bool = Field(
        description="특정 개인이나 대상을 직접적으로 공격하거나 비하하는지 여부"
    )

    reason: Optional[str] = Field(
        default=None, description="악성 요소가 있다면 그 이유"
    )


llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

parses = PydanticOutputParser(pydantic_object=CommentModeration)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 SNS에 댓글을 분석하여 악성 댓글을 자동으로 필터링 해주는 AI야. \n\n{format_instructions}",
        ),
        ("human", "댓글 : {comment}"),
    ]
).partial(format_instructions=parses.get_format_instructions())

chain = prompt | llm | parses


comments = [
    "제품이 정말 마음에 들어요. 배송도 빠르고 포장도 깔끔했습니다.",
    "이딴 것도 상품이라고 파냐? 진짜 개같네.",
    "판매자님 설명도 제대로 안 읽고 물건 보내시나요? 일을 이렇게밖에 못하세요?",
    "제품은 괜찮은데 가격이 조금 비싼 것 같아요.",
    "서비스가 너무 엉망이네요. 담당자는 진짜 무능한 것 같습니다.",
    "와 진짜 존나 맛없어요. 돈 아까워 죽겠네.",
    "상품 자체는 괜찮지만 직원 응대가 너무 불친절했습니다.",
    "판매자 너는 장사할 자격이 없다. 머리가 있으면 이런 식으로 운영하지 마라.",
]

for comment in comments:
    res = chain.invoke({"comment": comment})
    print(f"'{comment}")
    print(f"toxicity : {res.toxicity}")
    print(f"욕설 포함 여부 : {res.contains_profanity}")
    print(f"인신 공격 여부 : {res.contains_personal_attack}")
    if res.reason:
        print(f"문제가 있을 때 그 이유 : {res.reason}")
    print()

'제품이 정말 마음에 들어요. 배송도 빠르고 포장도 깔끔했습니다.
toxicity : safe
욕설 포함 여부 : False
인신 공격 여부 : False

'이딴 것도 상품이라고 파냐? 진짜 개같네.
toxicity : toxic
욕설 포함 여부 : True
인신 공격 여부 : False
문제가 있을 때 그 이유 : 욕설 및 비속어 사용과 상품에 대한 모욕

'판매자님 설명도 제대로 안 읽고 물건 보내시나요? 일을 이렇게밖에 못하세요?
toxicity : toxic
욕설 포함 여부 : False
인신 공격 여부 : True
문제가 있을 때 그 이유 : 판매자에 대한 직접적 비하의 개인 공격이 포함되어 있음

'제품은 괜찮은데 가격이 조금 비싼 것 같아요.
toxicity : safe
욕설 포함 여부 : False
인신 공격 여부 : False

'서비스가 너무 엉망이네요. 담당자는 진짜 무능한 것 같습니다.
toxicity : toxic
욕설 포함 여부 : False
인신 공격 여부 : True
문제가 있을 때 그 이유 : 담당자에 대한 직접적이고 모욕적인 비난으로 개인 공격에 해당

'와 진짜 존나 맛없어요. 돈 아까워 죽겠네.
toxicity : warning
욕설 포함 여부 : True
인신 공격 여부 : False
문제가 있을 때 그 이유 : 강한 욕설 및 부정적 표현으로 인한 악성 요소 판단

'상품 자체는 괜찮지만 직원 응대가 너무 불친절했습니다.
toxicity : warning
욕설 포함 여부 : False
인신 공격 여부 : False
문제가 있을 때 그 이유 : 직원 응대의 불친절함에 대한 불만

'판매자 너는 장사할 자격이 없다. 머리가 있으면 이런 식으로 운영하지 마라.
toxicity : toxic
욕설 포함 여부 : False
인신 공격 여부 : True
문제가 있을 때 그 이유 : 판매자를 향한 직접적인 모욕적 비판 및 개인 공격



## 과제 B : 식당 리뷰 -> 메뉴별 평가 추출


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class MenuRating(BaseModel):
    name: str = Field(description="메뉴 이름")

    rating: int = Field(description="메뉴 평점. 1점부터 5점까지의 정수", ge=1, le=5)

    comment: str = Field(description="해당 메뉴에 대한 구체적인 평가 내용")


class RestaurantReview(BaseModel):
    overall_rating: int = Field(
        description="레스토랑 전체 평점. 1점부터 5점까지의 정수", ge=1, le=5
    )

    menus: list[MenuRating] = Field(description="리뷰에서 언급된 메뉴별 평가 목록")

    would_revisit: bool = Field(
        description="고객이 해당 레스토랑을 다시 방문할 의향이 있는지 여부"
    )


parses = PydanticOutputParser(pydantic_object=RestaurantReview)


llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            너는 식당 리뷰를 분석하는 AI야.

            리뷰에서 다음 정보를 추출해.

            1. 전체 레스토랑 평점
            2. 언급된 모든 메뉴의 이름, 평점, 평가
            3. 고객의 재방문 의사

            반드시 아래 형식 지침을 따라 JSON으로만 응답해.
            다른 설명이나 문장은 작성하지 마.

            {format_instructions}
            """,
        ),
        ("human", "{review}"),
    ]
).partial(format_instructions=parses.get_format_instructions())

chain = prompt | llm | parses

reviews = [
    "전체적으로 정말 만족스러운 식사였습니다. 별점은 5점입니다. 김치찌개는 국물이 얼큰하고 깊은 맛이 나서 정말 맛있었고 5점 주고 싶어요. 제육볶음도 고기가 부드럽고 양념이 맛있어서 만족스러웠습니다. 계란말이는 특별한 맛은 아니었지만 무난했습니다. 다음에 근처에 오면 다시 방문할 것 같아요.",
    "전체 평점은 3점 정도입니다. 파스타는 면이 너무 많이 익어서 아쉬웠고 소스도 조금 짰습니다. 그래도 피자는 도우가 바삭하고 치즈가 맛있어서 4점을 주고 싶습니다. 샐러드는 신선하고 괜찮았어요. 음식마다 편차가 있는 것 같습니다. 굳이 다시 찾아올 정도는 아니지만 근처에 있다면 다시 갈 수도 있을 것 같네요.",
    "여기는 진짜 재방문하고 싶은 맛집입니다. 전체적으로 5점입니다. 돈까스는 튀김옷이 바삭하고 고기도 두툼해서 최고였어요. 냉모밀은 국물이 시원하고 면도 쫄깃해서 돈까스와 같이 먹기 좋았습니다. 우동은 평범해서 3점 정도지만 나쁘지는 않았습니다.",
    "기대하고 갔는데 생각보다 실망했습니다. 전체 평점은 2점입니다. 불고기는 고기가 너무 질기고 양념도 지나치게 달아서 별로였어요. 된장찌개는 간이 너무 세서 먹기 힘들었습니다. 공깃밥은 괜찮았지만 밥만 먹으러 갈 곳은 아닌 것 같습니다. 다시 방문할 생각은 없습니다.",
    "친구 추천으로 방문했는데 꽤 괜찮았습니다. 전체적으로 4점 정도예요. 마라탕은 재료가 다양하고 국물도 맛있었습니다. 꿔바로우는 바삭하긴 한데 소스가 너무 달아서 3점입니다. 볶음밥은 간이 적당하고 양도 많아서 만족했습니다. 다음에 다른 메뉴도 먹어보고 싶어서 재방문할 것 같습니다.",
    "음식은 전반적으로 맛있었습니다. 전체 평점은 4점입니다. 치킨은 겉은 바삭하고 속은 촉촉해서 정말 맛있었어요. 감자튀김도 바삭해서 좋았습니다. 다만 떡볶이는 너무 맵고 짜서 제 취향에는 맞지 않았습니다. 그래도 치킨이 맛있어서 다시 방문할 의향이 있습니다.",
]


for review in reviews:
    res = chain.invoke({"review": review})
    print(f"전체 평점 : {res.overall_rating}")
    print(f"재방문 의사 : {'있음' if res.would_revisit else '없음'}")
    print()

    for menu in res.menus:
        print(f"메뉴 이름 : {menu.name}")
        print(f"메뉴 이름 : {menu.rating} / 5")
        print(f"메뉴 이름 : {menu.comment}")
        print()

    print()

전체 평점 : 5
재방문 의사 : 있음

메뉴 이름 : 김치찌개
메뉴 이름 : 5 / 5
메뉴 이름 : 국물이 얼큰하고 깊은 맛이 나서 정말 맛있었고

메뉴 이름 : 제육볶음
메뉴 이름 : 4 / 5
메뉴 이름 : 고기가 부드럽고 양념이 맛있어서 만족스러웠습니다.

메뉴 이름 : 계란말이
메뉴 이름 : 3 / 5
메뉴 이름 : 특별한 맛은 없었지만 무난했습니다.


전체 평점 : 3
재방문 의사 : 있음

메뉴 이름 : 파스타
메뉴 이름 : 2 / 5
메뉴 이름 : 면이 너무 많이 익고 소스도 짜서 아쉬웠습니다.

메뉴 이름 : 피자
메뉴 이름 : 4 / 5
메뉴 이름 : 도우가 바삭하고 치즈가 맛있어서 4점을 주고 싶습니다.

메뉴 이름 : 샐러드
메뉴 이름 : 3 / 5
메뉴 이름 : 샐러드는 신선하고 괜찮았어요.


전체 평점 : 5
재방문 의사 : 있음

메뉴 이름 : 돈까스
메뉴 이름 : 5 / 5
메뉴 이름 : 튀김옷이 바삭하고 고기도 두툼해서 최고였어요.

메뉴 이름 : 냉모밀
메뉴 이름 : 5 / 5
메뉴 이름 : 국물이 시원하고 면도 쫄깃해서 돈까스와 같이 먹기 좋았습니다.

메뉴 이름 : 우동
메뉴 이름 : 3 / 5
메뉴 이름 : 평범해서 3점 정도지만 나쁘지는 않았습니다.


전체 평점 : 2
재방문 의사 : 없음

메뉴 이름 : 불고기
메뉴 이름 : 2 / 5
메뉴 이름 : 고기가 너무 질기고 양념도 지나치게 달아서 별로였다.

메뉴 이름 : 된장찌개
메뉴 이름 : 2 / 5
메뉴 이름 : 간이 너무 세서 먹기 힘들었습니다.

메뉴 이름 : 공깃밥
메뉴 이름 : 3 / 5
메뉴 이름 : 공깃밥은 괜찮았지만 밥만 먹으러 갈 곳은 아닌 것 같습니다.


전체 평점 : 4
재방문 의사 : 있음

메뉴 이름 : 마라탕
메뉴 이름 : 4 / 5
메뉴 이름 : 재료가 다양하고 국물도 맛있었습니다.

메뉴 이름 : 꿔바로우
메뉴 이름 : 3 / 5
메뉴 이름 : 바삭하긴 한데 소스가 너무 달아서 아쉽다.

메뉴 이름 : 볶음밥
메뉴 이름 : 4 / 

## 과제 C : 일정 정보 추출


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Optional


class Meeting(BaseModel):
    title: str = Field(description="회의 제목")

    date: str = Field(description="회의 날짜. YYYY-MM-DD 형식")

    time: str = Field(description="회의 시간. HH:MM 형식")

    location: Optional[str] = Field(default=None, description="회의 장소")

    attendees: list[str] = Field(description="회의 참석자 이름 목록")


llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

parses = PydanticOutputParser(pydantic_object=Meeting)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
                너는 날짜, 시간, 장소를 추출해서
                캘린더 등록용 정형 데이터를 만드는 AI야
                \n\n{format_instructions}"
                """,
        ),
        ("human", "{input}"),
    ]
).partial(format_instructions=parses.get_format_instructions())

chain = prompt | llm | parses

inputs = [
    "내일 오후 3시에 강남역 스타벅스에서 김민지 매니저랑 프로젝트 미팅",
    "12월 20일 오전 10시에 판교 본사 회의실에서 개발팀 이준호님, 박서연님과 주간 회의",
    "다음 주 화요일 오후 2시에 온라인으로 최현우 팀장님이랑 신규 서비스 기획 회의",
    "2025년 12월 24일 오후 4시 서울역 근처 카페에서 김민지, 이준호와 연말 프로젝트 회의",
    "금요일 오전 11시에 강남 사무실 3층 회의실에서 박서연 매니저와 면접 일정 조율 미팅",
    "오늘 저녁 7시에 홍대입구역 스타벅스에서 친구 김민수, 이지은이랑 팀 프로젝트 회의",
    "다음 달 5일 오후 1시 30분에 회사 대회의실에서 개발팀 전체와 신규 기능 출시 회의",
]

for input in inputs:
    res = chain.invoke({"input": input})
    print(f"제목 : {res.title}")
    print(f"날짜 및 시간 {res.date} {res.time}")
    print(f"장소 : {res.location}")
    print(f"참석자 : {', '.join(res.attendees)}")
    print()

제목 : 프로젝트 미팅
날짜 및 시간 2026-09-12 15:00
장소 : 강남역 스타벅스
참석자 : 김민지 매니저

제목 : 주간 회의
날짜 및 시간 2026-12-20 10:00
장소 : 판교 본사 회의실
참석자 : 이준호님, 박서연님

제목 : 신규 서비스 기획 회의
날짜 및 시간 2026-09-15 14:00
장소 : 온라인
참석자 : 최현우 팀장님

제목 : 연말 프로젝트 회의
날짜 및 시간 2025-12-24 16:00
장소 : 서울역 근처 카페
참석자 : 김민지, 이준호

제목 : 면접 일정 조율 미팅
날짜 및 시간 2026-09-11 11:00
장소 : 강남 사무실 3층 회의실
참석자 : 박서연 매니저

제목 : 팀 프로젝트 회의
날짜 및 시간 2026-09-11 19:00
장소 : 홍대입구역 스타벅스
참석자 : 김민수, 이지은

제목 : 개발팀 전체와 신규 기능 출시 회의
날짜 및 시간 2026-10-05 13:30
장소 : 회사 대회의실
참석자 : 개발팀 전체

